# Notebook 04: JEPA-Based VLA Models — The ICRA-Relevant Frontier

**Goal:** Deep dive into two 2026 papers that integrate JEPA with VLAs — your key references for an ICRA paper.

---

## Two Approaches to JEPA + VLA

| | JEPA-VLA (Tsinghua) | VLA-JEPA (USTC) |
|---|---|---|
| **Paper** | arXiv:2602.11832 | arXiv:2602.10098 |
| **Core idea** | Plug V-JEPA 2 features INTO existing VLAs | Build VLA WITH JEPA world model |
| **V-JEPA 2 role** | Frozen feature extractor | Frozen supervision target |
| **VLM backbone** | Any (tested Chameleon, OpenVLA-OFT) | Qwen3-VL-2B |
| **World model** | None | 12-layer transformer |
| **Action head** | Existing VLA heads | Flow-matching DiT-B |
| **Code** | Not released | [github.com/ginwind/VLA-JEPA](https://github.com/ginwind/VLA-JEPA) |
| **LIBERO avg** | ~96% | **97.2%** |

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)

## 0.5 The 5W+H of JEPA-Based VLAs

### WHO developed these approaches?
- **JEPA-VLA:** Tsinghua University team (arXiv:2602.11832, Feb 2026)
- **VLA-JEPA:** University of Science and Technology of China (USTC) team (arXiv:2602.10098, Feb 2026)

Both appeared within days of each other, independently exploring the same question: how to integrate JEPA with VLAs.

### WHAT are VLAs (Vision-Language-Action models)?

VLAs combine three modalities to produce robot actions:

| Component | Input | Output | Examples |
|-----------|-------|--------|----------|
| **Vision encoder** | Camera images | Visual tokens | CLIP, SigLIP, DINOv2 |
| **Language model** | Text instructions | Semantic understanding | Llama, Qwen, GPT |
| **Action head** | Visual + language features | Robot actions | MLP, Diffusion, Flow matching |

**The VLA pipeline:**
$$a_{0:H} = \text{ActionHead}(\text{VLM}(\text{VisionEnc}(I_t), \text{TokenEmbed}(l)))$$

### WHERE does JEPA improve VLAs?

Standard VLAs use **static image encoders** (CLIP, SigLIP) that:
- Were trained on image-text pairs (not video)
- Don't understand temporal dynamics
- Lack physical intuition about action consequences

V-JEPA 2 adds what's missing:
1. **Temporal understanding:** Trained on video, captures motion and dynamics
2. **Physical priors:** Predicts future representations → understands physics
3. **Policy alignment:** V-JEPA 2-AC was fine-tuned on robot data → knows what actions do

### WHEN did the JEPA + VLA integration emerge?

| Date | Event |
|------|-------|
| Jun 2023 | RT-2: First VLA (Google) |
| Oct 2024 | OpenVLA: Open-source VLA |
| Nov 2024 | pi0: Flow matching for VLA actions |
| Jan 2025 | V-JEPA 2 released (Meta) |
| Feb 2026 | JEPA-VLA and VLA-JEPA papers (independent) |
| **Now** | **Your paper: Unified JEPA-VLA?** |

### WHY are there two different approaches?

**JEPA-VLA (plug-in):** Uses V-JEPA 2 as a **better vision backbone**
- Advantage: Can enhance ANY existing VLA
- Limitation: No world model for planning

**VLA-JEPA (end-to-end):** Uses V-JEPA 2 as a **supervision target** for a world model
- Advantage: World model enables latent planning
- Limitation: Custom architecture, not easily pluggable

### HOW do they compare mathematically?

**JEPA-VLA loss (plug-in, action prediction only):**
$$\mathcal{L}_{\text{JEPA-VLA}} = \mathcal{L}_{\text{action}}(\pi_\theta(V(I_t) \oplus F(I_{t-1:t}), l), \; a_t)$$

where $V$ is the VLA's vision encoder, $F$ is frozen V-JEPA 2, and $\oplus$ is fusion (concat or cross-attention).

**VLA-JEPA loss (end-to-end, action + world model):**
$$\mathcal{L}_{\text{VLA-JEPA}} = \underbrace{\mathcal{L}_{\text{FM}}}_{\text{flow matching}} + \beta \underbrace{\mathcal{L}_{\text{WM}}}_{\text{world model}}, \quad \beta = 0.1$$

$$\mathcal{L}_{\text{WM}} = \sum_{k=1}^T \| P_\theta^{\text{WM}}(s_{t_0:k}, z_{t_0:k}) - \text{sg}(F(I_{t_k})) \|_1$$

$$\mathcal{L}_{\text{FM}} = \mathbb{E}_{t \sim U[0,1]} \left[ \| v_\theta(a_t, t \,|\, z_a) - (a_{\text{gt}} - \epsilon) \|_2^2 \right]$$

## Part A: JEPA-VLA — Plugging V-JEPA 2 Into Existing VLAs

### The Problem with Current VLAs

Current VLAs use vision backbones (CLIP, DINOv2, SigLIP) trained on:
- Image classification
- Image-text contrastive learning

These backbones are **suboptimal for robotics** because:
1. They don't understand temporal dynamics (how actions affect the world)
2. They lack "policy priors" — knowledge about what actions lead to what outcomes
3. They're trained on static images, not videos

**V-JEPA 2 fixes this** — it's trained on video, understands physics, and has built-in temporal reasoning.

### JEPA-VLA Architecture

```
Standard VLA:
  Camera Image → [CLIP/DINOv2] → VLM → Action

JEPA-VLA:
  Camera Image (t-1, t) → [Frozen V-JEPA 2] → h_t
  Camera Image → [CLIP/DINOv2] → visual tokens
                                        ↓
  Language instruction → [VLM] + h_t → Action
                               ↑
                    V-JEPA 2 features added via:
                    - Early Fusion (concatenation) OR
                    - Gated Cross-Attention
```

In [ ]:
class EarlyFusion(nn.Module):
    """JEPA-VLA Early Fusion: concatenate V-JEPA 2 features with VLA tokens.
    
    Used for non-pretrained VLAs (e.g., training from scratch).
    The V-JEPA 2 features are projected and prepended to the token sequence.
    """
    def __init__(self, jepa_dim=1408, vla_dim=768):
        super().__init__()
        self.proj = nn.Linear(jepa_dim, vla_dim)  # align dimensions
    
    def forward(self, vla_tokens, jepa_features):
        """
        vla_tokens:     [B, N_vla, D_vla] — from standard VLA backbone
        jepa_features:  [B, N_jepa, D_jepa] — from frozen V-JEPA 2
        """
        h = self.proj(jepa_features)  # [B, N_jepa, D_vla]
        return torch.cat([h, vla_tokens], dim=1)  # prepend JEPA features


class GatedCrossAttention(nn.Module):
    """JEPA-VLA Gated Fusion: cross-attention from VLA to V-JEPA 2 features.
    
    Used for pretrained VLAs (e.g., OpenVLA-OFT) to avoid disrupting
    the pretrained representations.
    
    Inserted every 8 transformer decoder layers.
    VLA tokens are queries, V-JEPA 2 features are keys/values.
    """
    def __init__(self, vla_dim=768, jepa_dim=1408, num_heads=8):
        super().__init__()
        self.norm = nn.LayerNorm(vla_dim)
        self.q_proj = nn.Linear(vla_dim, vla_dim)
        self.kv_proj = nn.Linear(jepa_dim, vla_dim * 2)
        self.out_proj = nn.Linear(vla_dim, vla_dim)
        self.gate = nn.Parameter(torch.zeros(1))  # learnable gate, init=0
        self.num_heads = num_heads
        self.head_dim = vla_dim // num_heads
    
    def forward(self, vla_tokens, jepa_features):
        """
        vla_tokens:    [B, N_vla, D_vla] — queries
        jepa_features: [B, N_jepa, D_jepa] — keys and values
        """
        B, N_q, D = vla_tokens.shape
        N_kv = jepa_features.shape[1]
        
        q = self.q_proj(self.norm(vla_tokens))  # [B, N_vla, D]
        kv = self.kv_proj(jepa_features)  # [B, N_jepa, 2*D]
        k, v = kv.chunk(2, dim=-1)
        
        # Reshape for multi-head attention
        q = q.view(B, N_q, self.num_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, N_kv, self.num_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, N_kv, self.num_heads, self.head_dim).transpose(1, 2)
        
        attn = F.scaled_dot_product_attention(q, k, v)
        attn = attn.transpose(1, 2).reshape(B, N_q, D)
        attn = self.out_proj(attn)
        
        # Gated residual: gate starts at 0, so initially JEPA features have no effect
        # As training progresses, the gate opens to incorporate JEPA information
        return vla_tokens + torch.tanh(self.gate) * attn


# Demo
B = 2
vla_tokens = torch.randn(B, 50, 768)   # 50 VLA tokens
jepa_feats = torch.randn(B, 512, 1408)  # 512 V-JEPA 2 tokens (2 frames × 256 patches)

# Early fusion
early = EarlyFusion(jepa_dim=1408, vla_dim=768)
fused_early = early(vla_tokens, jepa_feats)
print(f"Early Fusion:")
print(f"  VLA tokens: {vla_tokens.shape} + JEPA features: {jepa_feats.shape}")
print(f"  → Fused: {fused_early.shape}  (JEPA tokens prepended)")
print()

# Gated cross-attention
gated = GatedCrossAttention(vla_dim=768, jepa_dim=1408)
fused_gated = gated(vla_tokens, jepa_feats)
print(f"Gated Cross-Attention:")
print(f"  VLA tokens: {vla_tokens.shape} (queries)")
print(f"  JEPA features: {jepa_feats.shape} (keys/values)")
print(f"  → Output: {fused_gated.shape}  (same size, JEPA info added via gated attention)")
print(f"  Gate value (init): {gated.gate.item():.4f} → tanh(gate) = {torch.tanh(gated.gate).item():.4f}")
print(f"  (Gate starts at 0 = no JEPA influence, learned during training)")

### Leakage-Free Design: The Critical Innovation in VLA-JEPA

**The leakage problem:** If the VLM can see future frames, it can learn to **copy** rather than **predict**. The world model becomes useless because the VLM already has the answer.

**VLA-JEPA's solution — strict information separation:**

```
STUDENT pathway (trainable):           TARGET pathway (frozen):
─────────────────────────────          ──────────────────────────
VLM sees ONLY:                         V-JEPA 2 encoder sees:
  • Current frame I_{t_0}               • ALL frames I_{t_0}, ..., I_{t_T}
  • Language instruction l               
                                        Outputs: s_{t_1:T} (targets)
Outputs: latent actions z_{t_0:T}       ← stop-gradient! ←
         ↓
World model predicts: ŝ_{t_1:T}
         ↓
L1 loss: ‖ŝ_{t_1:T} - s_{t_1:T}‖₁
```

**Why this matters:**
- The VLM CANNOT cheat — it only sees the current frame
- To produce good latent actions $z$, the VLM must learn to **plan ahead**
- The world model must accurately predict consequences of these latent actions
- This forces the system to develop **genuine planning capability**

**Mathematical guarantee:** Information flow is one-way:
$$z_{t_i} = f_\theta^{\text{VLM}}(I_{t_0}, l) \quad \text{(no future frame access)}$$
$$\hat{s}_{t_k} = f_\theta^{\text{WM}}(s_{t_0:k-1}, z_{t_0:k-1}) \quad \text{(autoregressive, no peeking)}$$
$$s_{t_k} = \text{sg}(F(I_{t_k})) \quad \text{(stop-gradient blocks backward flow)}$$

## Part B: VLA-JEPA — Building VLA WITH a JEPA World Model

### The Key Innovation: Leakage-Free State Prediction

**Problem:** If the VLM sees future frame information, it can "cheat" and ignore the world model.

**Solution:** Structural separation:
- **Target pathway:** Frozen V-JEPA 2 encodes future frames → supervision targets (stop-gradient)
- **Student pathway:** VLM sees ONLY current frame → must learn to predict future via world model

### Architecture

```
                         ┌─────────────────────┐
Current frame + Lang ──► │  Qwen3-VL-2B (VLM)  │──► <latent> tokens (z_t)
                         └─────────────────────┘        │
                                                        ▼
                         ┌─────────────────────┐   ┌──────────┐
Future frames ──────────►│ Frozen V-JEPA 2 Enc │──►│  Targets │ (stop-grad)
                         └─────────────────────┘   └────┬─────┘
                                                        │
          z_t (latent actions) + past states ──►  World Model Predictor
                                                        │
                                                   ŝ_{t+1:T}
                                                        │
                                              L1 loss vs targets
```

### Mathematical Formulation

**Eq 1 — World state encoding (multi-view):**
$$s_{t_i} = \|_v F(I_{v,t_i})$$
where $F$ is frozen V-JEPA 2, $\|$ is concatenation across camera views.

**Eq 2 — Latent action from VLM:**
$$z_{t_i} = p_\theta^{\text{VLM}}(\langle\text{latent}_i\rangle | \{I_{j,t_0}\}_{j=0}^v, l)$$

**Eq 3 — Autoregressive world model prediction:**
$$\hat{s}_{t_1:i+1} = p_\theta^{\text{WM}}(s_{t_0:i}, z_{t_0:i})$$

**Eq 5 — World modeling loss:**
$$\mathcal{L}_{\text{WM}} = \sum_{k=1}^T (\hat{s}_{t_k} - s_{t_k})$$

**Eq 8 — Flow matching for actions:**
$$\mathcal{L}_{\text{FM}} = \mathbb{E}[\|v_\theta(a_t, t | z_a) - (a_{0:H} - \epsilon)\|_2^2]$$

**Eq 9 — Total loss:**
$$\mathcal{L} = \mathcal{L}_{\text{FM}} + \beta \cdot \mathcal{L}_{\text{WM}}, \quad \beta = 0.1$$

In [ ]:
class MiniVLAJEPA(nn.Module):
    """Simplified VLA-JEPA architecture.
    
    Real implementation: refs/VLA-JEPA/ (built on starVLA framework)
    
    Components:
    1. Frozen V-JEPA 2 encoder → world state encoding
    2. VLM (Qwen3-VL-2B) → latent actions from current obs + language
    3. World model predictor → future state prediction
    4. Flow-matching action head → robot actions
    """
    def __init__(self, state_dim=256, latent_dim=64, action_dim=7, horizon=8):
        super().__init__()
        self.horizon = horizon
        
        # 1. Frozen V-JEPA 2 encoder (simulated as fixed random projection)
        self.vjepa2_encoder = nn.Linear(3 * 64 * 64, state_dim)  # simplified
        for p in self.vjepa2_encoder.parameters():
            p.requires_grad = False  # FROZEN
        
        # 2. VLM → latent actions (simulated)
        self.vlm_latent_head = nn.Linear(state_dim + 32, latent_dim * horizon)  # obs + lang → latent actions
        
        # 3. World model predictor (12-layer transformer, simplified as MLP)
        self.world_model = nn.Sequential(
            nn.Linear(state_dim + latent_dim, state_dim * 2),
            nn.GELU(),
            nn.Linear(state_dim * 2, state_dim),
        )
        
        # 4. Flow-matching action head (DiT-B, simplified)
        self.action_head = nn.Sequential(
            nn.Linear(latent_dim + 8, 128),  # latent action + proprioception
            nn.GELU(),
            nn.Linear(128, action_dim),
        )
    
    def encode_state(self, frames):
        """Eq 1: Encode frames with frozen V-JEPA 2."""
        B, T = frames.shape[:2]
        flat = frames.view(B * T, -1)
        with torch.no_grad():
            states = self.vjepa2_encoder(flat)
        return states.view(B, T, -1)
    
    def get_latent_actions(self, current_obs, language):
        """Eq 2: Generate latent actions from VLM."""
        combined = torch.cat([current_obs, language], dim=-1)
        z = self.vlm_latent_head(combined)  # [B, latent_dim * horizon]
        return z.view(-1, self.horizon, z.shape[-1] // self.horizon)
    
    def predict_future_states(self, initial_state, latent_actions):
        """Eq 3: Autoregressive world model prediction."""
        B = initial_state.shape[0]
        predictions = []
        current = initial_state
        
        for t in range(latent_actions.shape[1]):
            combined = torch.cat([current, latent_actions[:, t]], dim=-1)
            next_state = self.world_model(combined)
            predictions.append(next_state)
            current = next_state
        
        return torch.stack(predictions, dim=1)  # [B, T, state_dim]
    
    def forward(self, frames, language, proprioception, gt_actions):
        B = frames.shape[0]
        
        # Step 1: Encode all frames with frozen V-JEPA 2 (targets)
        gt_states = self.encode_state(frames)  # [B, T, state_dim] — stop gradient
        
        # Step 2: VLM generates latent actions from current obs only
        current_obs = gt_states[:, 0]  # only first frame!
        latent_actions = self.get_latent_actions(current_obs, language)
        
        # Step 3: World model predicts future states
        predicted_states = self.predict_future_states(gt_states[:, 0], latent_actions)
        
        # Step 4: World model loss (Eq 5)
        T_pred = min(predicted_states.shape[1], gt_states.shape[1] - 1)
        wm_loss = F.l1_loss(predicted_states[:, :T_pred], gt_states[:, 1:T_pred+1])
        
        # Step 5: Action loss via flow matching (Eq 8, simplified as MSE)
        action_input = torch.cat([latent_actions[:, :gt_actions.shape[1]], 
                                   proprioception.unsqueeze(1).expand(-1, gt_actions.shape[1], -1)], dim=-1)
        pred_actions = self.action_head(action_input)
        action_loss = F.mse_loss(pred_actions, gt_actions)
        
        # Step 6: Total loss (Eq 9)
        beta = 0.1
        total_loss = action_loss + beta * wm_loss
        
        return {
            'total_loss': total_loss,
            'action_loss': action_loss,
            'wm_loss': wm_loss,
        }


# Demo
model = MiniVLAJEPA(state_dim=128, latent_dim=32, action_dim=7, horizon=8)

# Simulated inputs
B = 4
frames = torch.randn(B, 8, 3, 64, 64)      # 8 frames
language = torch.randn(B, 32)                 # language embedding
proprioception = torch.randn(B, 8)            # robot state
gt_actions = torch.randn(B, 7, 7)            # ground truth actions

losses = model(frames, language, proprioception, gt_actions)

print("VLA-JEPA Forward Pass")
print("=" * 40)
print(f"Action loss (L_FM):  {losses['action_loss'].item():.4f}")
print(f"World model loss (L_WM): {losses['wm_loss'].item():.4f}")
print(f"Total loss (L_FM + 0.1 * L_WM): {losses['total_loss'].item():.4f}")
print(f"")
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"Frozen params:    {sum(p.numel() for p in model.parameters() if not p.requires_grad):,}")

In [ ]:
# FLOW MATCHING: Complete mathematical derivation with visualization

def flow_matching_deep_dive():
    """
    Flow matching learns a velocity field that transports noise → actions.
    
    Core idea: Define a straight-line path from noise ε to target action a_gt:
        a_t = (1-t)·ε + t·a_gt    for t ∈ [0, 1]
    
    The velocity along this path is constant:
        da/dt = a_gt - ε   (the derivative of (1-t)·ε + t·a_gt w.r.t. t)
    
    Train a neural network v_θ to predict this velocity:
        L = E_{t,ε} [ ‖v_θ(a_t, t | z) - (a_gt - ε)‖² ]
    
    At inference, integrate from t=0 (noise) to t=1 (action) using Euler steps.
    """
    
    action_dim = 2  # 2D for visualization
    n_trajectories = 50
    
    # Ground truth action (target)
    a_gt = torch.tensor([0.5, 0.3])
    
    # Sample noise points
    torch.manual_seed(42)
    epsilons = torch.randn(n_trajectories, action_dim) * 0.3
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Plot 1: The straight-line interpolation paths
    ax = axes[0]
    timesteps_plot = torch.linspace(0, 1, 20)
    
    for i in range(min(15, n_trajectories)):
        trajectory_x = []
        trajectory_y = []
        for t_val in timesteps_plot:
            a_t = (1 - t_val) * epsilons[i] + t_val * a_gt
            trajectory_x.append(a_t[0].item())
            trajectory_y.append(a_t[1].item())
        
        ax.plot(trajectory_x, trajectory_y, 'b-', alpha=0.2, linewidth=0.5)
        ax.plot(trajectory_x[0], trajectory_y[0], 'ko', markersize=3, alpha=0.5)
    
    ax.plot(a_gt[0], a_gt[1], 'r*', markersize=20, zorder=10, label='Target action a_gt')
    ax.scatter(epsilons[:15, 0], epsilons[:15, 1], c='blue', s=20, alpha=0.5, label='Noise ε ~ N(0,I)')
    ax.set_xlabel('Action dim 1')
    ax.set_ylabel('Action dim 2')
    ax.set_title('Flow Matching: Straight-line paths\nfrom noise ε to target a_gt')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    # Plot 2: Velocity field at t=0.5
    ax = axes[1]
    grid_x = torch.linspace(-1, 1, 10)
    grid_y = torch.linspace(-0.5, 0.8, 10)
    gx, gy = torch.meshgrid(grid_x, grid_y, indexing='ij')
    
    # At any point a_t, the velocity field points toward a_gt
    # v* = a_gt - ε, but ε = (a_t - t·a_gt) / (1-t) for t ≠ 1
    t_vis = 0.5
    vx = (a_gt[0] - gx) / (1.0 - t_vis + 1e-6)  # simplified direction
    vy = (a_gt[1] - gy) / (1.0 - t_vis + 1e-6)
    
    # Normalize for visualization
    magnitude = torch.sqrt(vx**2 + vy**2).clamp(min=1e-6)
    vx_norm = vx / magnitude * 0.15
    vy_norm = vy / magnitude * 0.15
    
    ax.quiver(gx.numpy(), gy.numpy(), vx_norm.numpy(), vy_norm.numpy(), 
              magnitude.numpy(), cmap='coolwarm', scale=3)
    ax.plot(a_gt[0], a_gt[1], 'r*', markersize=20, zorder=10)
    ax.set_xlabel('Action dim 1')
    ax.set_ylabel('Action dim 2')
    ax.set_title(f'Learned Velocity Field at t={t_vis}\n(all arrows point toward target)')
    ax.grid(True, alpha=0.3)
    
    # Plot 3: Euler integration (inference)
    ax = axes[2]
    n_euler_steps_list = [1, 2, 4, 8]
    colors = ['#e74c3c', '#f39c12', '#2ecc71', '#3498db']
    
    epsilon_demo = torch.tensor([-0.3, -0.2])
    
    for n_steps, color in zip(n_euler_steps_list, colors):
        dt = 1.0 / n_steps
        current = epsilon_demo.clone()
        path_x = [current[0].item()]
        path_y = [current[1].item()]
        
        for step in range(n_steps):
            t_step = step * dt
            # True velocity at this point (in practice, the neural net predicts this)
            # v* = a_gt - ε where ε = (a_t - t·a_gt)/(1-t)
            velocity = (a_gt - current) / (1 - t_step + 1e-6)
            velocity = velocity.clamp(-5, 5)  # stability
            current = current + dt * velocity
            path_x.append(current[0].item())
            path_y.append(current[1].item())
        
        ax.plot(path_x, path_y, 'o-', color=color, markersize=4, linewidth=2,
                label=f'{n_steps} steps (Δt={dt:.2f})')
    
    ax.plot(a_gt[0], a_gt[1], 'r*', markersize=20, zorder=10, label='Target')
    ax.plot(epsilon_demo[0], epsilon_demo[1], 'ko', markersize=10, zorder=10, label='Start (noise)')
    ax.set_xlabel('Action dim 1')
    ax.set_ylabel('Action dim 2')
    ax.set_title('Inference: Euler Integration\n(4 steps is sufficient for VLA-JEPA)')
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("Flow Matching Summary:")
    print("─────────────────────")
    print("Training: Learn velocity field v_θ(a_t, t | z) where:")
    print("  • a_t = (1-t)·ε + t·a_gt    (interpolated point)")
    print("  • target velocity = a_gt - ε  (constant along path)")
    print("  • L = E[‖v_θ(a_t, t | z) - (a_gt - ε)‖²]")
    print()
    print("Inference (4 Euler steps):")
    print("  • a₀ = ε ~ N(0, I)")
    print("  • For t = 0, 0.25, 0.5, 0.75:")
    print("      a_{t+Δt} = a_t + Δt · v_θ(a_t, t | z)")
    print("  • a₁ = final action to execute")
    print()
    print("Why flow matching over diffusion?")
    print("  • Straight-line paths → fewer steps needed (4 vs 50-1000)")
    print("  • Constant velocity → easier to learn")
    print("  • Same quality as DDPM with 100-250x fewer steps")

flow_matching_deep_dive()

## Part C: VLA-JEPA's Two-Stage Training

### Stage 1: JEPA Pretraining (Eq 2-5)
- **Data:** Something-Something-v2 (220K videos) + DROID (76K robot trajectories)
- **Goal:** Learn latent actions and world model from video
- **Duration:** 50K steps, batch 256, 8 GPUs
- **What's trained:** VLM latent head + world model predictor
- **What's frozen:** V-JEPA 2 encoder (always)

### Stage 2: Action Head Fine-Tuning (Eq 6-9)
- **Data:** LIBERO or target robot dataset
- **Goal:** Map latent actions → real robot actions via flow matching
- **Duration:** 30K steps
- **What's trained:** Action head + VLM (lower lr) + world model
- **Key:** Uses DiT-B (16 layers, 12 heads) with 4 denoising steps

In [ ]:
# GATED CROSS-ATTENTION: Detailed visualization of the gating mechanism

def visualize_gated_cross_attention():
    """Show how the learnable gate controls JEPA feature influence over training."""
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    
    # Plot 1: Gate evolution during training
    ax = axes[0]
    training_steps = np.linspace(0, 50000, 500)
    
    # Simulate gate parameter learning (starts at 0, gradually opens)
    gate_raw = -3 + training_steps / 50000 * 6  # from -3 to +3
    gate_effective = np.tanh(gate_raw)
    
    ax.plot(training_steps, gate_effective, 'b-', linewidth=2)
    ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    ax.fill_between(training_steps, 0, gate_effective, alpha=0.2, color='blue')
    ax.set_xlabel('Training Steps')
    ax.set_ylabel('tanh(gate)')
    ax.set_title('Gated Cross-Attention:\nGate opens gradually during training')
    ax.annotate('JEPA features\nhave NO influence', xy=(5000, 0.02), fontsize=9, 
                color='red', ha='center')
    ax.annotate('JEPA features\nFULLY integrated', xy=(45000, 0.85), fontsize=9, 
                color='green', ha='center')
    ax.grid(True, alpha=0.3)
    ax.set_ylim(-0.2, 1.1)
    
    # Plot 2: Cross-attention pattern (VLA queries attending to JEPA keys)
    ax = axes[1]
    torch.manual_seed(42)
    
    N_vla = 8     # VLA tokens (queries)
    N_jepa = 12   # JEPA tokens (keys/values)
    D_ca = 16
    
    q = torch.randn(N_vla, D_ca)
    k = torch.randn(N_jepa, D_ca)
    
    attn = (q @ k.T / (D_ca ** 0.5)).softmax(dim=-1)
    
    im = ax.imshow(attn.detach().numpy(), cmap='Oranges', aspect='auto')
    plt.colorbar(im, ax=ax)
    ax.set_xlabel('JEPA token (key/value)')
    ax.set_ylabel('VLA token (query)')
    ax.set_title('Cross-Attention Weights\nVLA tokens query JEPA features')
    
    vla_labels = ['VLA₀', 'VLA₁', 'VLA₂', 'VLA₃', 'VLA₄', 'VLA₅', 'VLA₆', 'VLA₇']
    jepa_labels = [f'J{i}' for i in range(N_jepa)]
    ax.set_yticks(range(N_vla))
    ax.set_yticklabels(vla_labels, fontsize=8)
    ax.set_xticks(range(N_jepa))
    ax.set_xticklabels(jepa_labels, fontsize=7)
    
    # Plot 3: Information flow comparison
    ax = axes[2]
    ax.axis('off')
    
    comparison = """
    EARLY FUSION (for new VLAs):
    ━━━━━━━━━━━━━━━━━━━━━━━━━━
    [JEPA₁, JEPA₂, ..., VLA₁, VLA₂, ...]
          ↓ concatenated sequence ↓
    Standard self-attention processes all
    
    ✓ Simple implementation
    ✓ Full bidirectional attention
    ✗ Increases sequence length → O(N²) cost
    ✗ Disrupts pretrained VLA representations
    
    
    GATED CROSS-ATTENTION (for pretrained VLAs):
    ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
    VLA tokens ──(Q)──►┐
                        ├──► cross-attn ──(×gate)──► add to VLA
    JEPA tokens ─(K,V)─►┘
    
    ✓ Preserves pretrained VLA representations
    ✓ Gate starts at 0 → gradual integration
    ✓ No sequence length increase
    ✓ Inserted every 8 layers (sparse)
    ✗ More parameters per fusion layer
    """
    
    ax.text(0.05, 0.95, comparison, fontfamily='monospace', fontsize=9,
            verticalalignment='top', transform=ax.transAxes,
            bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
    ax.set_title('Fusion Strategy Comparison')
    
    plt.tight_layout()
    plt.show()

visualize_gated_cross_attention()

In [ ]:
def flow_matching_demo():
    """Demonstrate flow matching for action generation (used in VLA-JEPA and pi0).
    
    Flow matching learns a velocity field v(a_t, t) that transforms
    noise ε ~ N(0,I) into actions a_{0:H} along a straight-line path:
    
    a_t = (1-t) * ε + t * a_{0:H}     (stochastic bridge, Eq 7)
    
    The model learns:
    L_FM = E[||v_θ(a_t, t | z_a) - (a_{0:H} - ε)||²]   (Eq 8)
    
    At inference, integrate from t=0 (noise) to t=1 (action) in ~4 steps.
    """
    action_dim = 7
    horizon = 7  # predict 7 future actions
    
    # Ground truth action chunk
    a_gt = torch.randn(1, horizon, action_dim) * 0.1  # small actions
    
    # Noise
    epsilon = torch.randn_like(a_gt)
    
    # Flow matching interpolation at different timesteps
    fig, axes = plt.subplots(1, 5, figsize=(15, 3))
    timesteps = [0.0, 0.25, 0.5, 0.75, 1.0]
    
    for ax, t in zip(axes, timesteps):
        # Eq 7: stochastic bridge
        a_t = (1 - t) * epsilon + t * a_gt
        ax.imshow(a_t[0].detach().numpy(), aspect='auto', cmap='RdBu', vmin=-0.5, vmax=0.5)
        ax.set_title(f't = {t:.2f}\n{"Pure noise" if t == 0 else "GT action" if t == 1 else "Interpolated"}')
        ax.set_xlabel('Action dim')
        ax.set_ylabel('Horizon step')
    
    plt.suptitle('Flow Matching: Noise → Action via straight-line interpolation', fontsize=13)
    plt.tight_layout()
    plt.show()
    
    # The velocity field to learn
    target_velocity = a_gt - epsilon  # (a_{0:H} - ε)
    print(f"Target velocity field: v* = a_gt - ε")
    print(f"Shape: {target_velocity.shape}  (same as action chunk)")
    print(f"")
    print(f"At inference (4 denoising steps):")
    print(f"  Start: a_0 = ε ~ N(0,I)")
    print(f"  Step 1: a_{0.25} = a_0 + 0.25 * v_θ(a_0, 0 | z_a)")
    print(f"  Step 2: a_{0.50} = a_{0.25} + 0.25 * v_θ(a_{0.25}, 0.25 | z_a)")
    print(f"  Step 3: a_{0.75} = a_{0.50} + 0.25 * v_θ(a_{0.50}, 0.50 | z_a)")
    print(f"  Step 4: a_{1.0} = a_{0.75} + 0.25 * v_θ(a_{0.75}, 0.75 | z_a) ← final action")

flow_matching_demo()

## Part D: Comparative Analysis for ICRA

### What Each Paper Contributes

**JEPA-VLA (plug-in approach):**
- Shows V-JEPA 2 features are **complementary** to existing VLA backbones
- Demonstrates consistent improvement across multiple VLA architectures
- Key finding: V-JEPA 2 provides "policy-aligned anticipatory knowledge" (knows what successful actions look like)

**VLA-JEPA (end-to-end approach):**
- Shows JEPA world model enables better action prediction
- Key innovation: leakage-free design prevents shortcutting
- Human video pretraining transfers: more human video → better robot performance
- Unique behaviors emerge (e.g., repeated grasping learned from human video)

### Benchmark Results

In [ ]:
# Benchmark comparison visualization
benchmarks = {
    'LIBERO-Spatial': {'OpenVLA-OFT': 96.0, 'JEPA-VLA': 98.0, 'VLA-JEPA': 97.6},
    'LIBERO-Object': {'OpenVLA-OFT': 97.2, 'JEPA-VLA': 99.2, 'VLA-JEPA': 99.6},
    'LIBERO-Goal': {'OpenVLA-OFT': 91.6, 'JEPA-VLA': 94.4, 'VLA-JEPA': 96.0},
    'LIBERO-Long': {'OpenVLA-OFT': 88.0, 'JEPA-VLA': 94.8, 'VLA-JEPA': 95.6},
}

methods = ['OpenVLA-OFT', 'JEPA-VLA', 'VLA-JEPA']
colors = ['#808080', '#4CAF50', '#2196F3']

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(benchmarks))
width = 0.25

for i, method in enumerate(methods):
    values = [benchmarks[b][method] for b in benchmarks]
    bars = ax.bar(x + i * width, values, width, label=method, color=colors[i])
    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
                f'{val}%', ha='center', va='bottom', fontsize=8)

ax.set_xlabel('Benchmark')
ax.set_ylabel('Success Rate (%)')
ax.set_title('JEPA-Based VLAs vs Baseline on LIBERO Benchmarks')
ax.set_xticks(x + width)
ax.set_xticklabels(benchmarks.keys(), rotation=15)
ax.legend()
ax.set_ylim(85, 102)
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

print("Key observations:")
print("1. Both JEPA-based methods improve over the OpenVLA-OFT baseline")
print("2. VLA-JEPA (end-to-end with world model) slightly outperforms JEPA-VLA (plug-in)")
print("3. Biggest gains on LONG-HORIZON tasks — world model helps with planning")
print("4. Near-ceiling on LIBERO-Object — JEPA features excel at object understanding")

## Part E: Open Research Gaps (ICRA Paper Opportunities)

### Gap 1: Combining Both Approaches
JEPA-VLA adds features, VLA-JEPA adds a world model. **Nobody has done both.**
Use V-JEPA 2 features as input AND as world model supervision targets.

### Gap 2: Safety Integration
Neither paper addresses safety. **CBF-JEPA** for safe humanoid VLA control.
Define CBFs in V-JEPA 2's latent space for guaranteed safe planning.

### Gap 3: Multi-Embodiment
Both papers test on single robot arms. V-JEPA 2-AC showed zero-shot transfer.
Can JEPA-VLA generalize across humanoid bodies?

### Gap 4: Real-Time Planning
V-JEPA 2-AC takes 16 seconds per action. For humanoid control you need <100ms.
Learned amortized planning in JEPA latent space.

### Gap 5: Humanoid-Specific
All current work is on arms. Humanoid locomotion + manipulation with JEPA world model.

---

**Next notebook:** Detailed ICRA research gap analysis and paper outline.